In [0]:
df = spark.table("cpg_planning.bronze.demand_statcan_retail")



In [0]:
# Shape of the data
print(f"Rows: {df.count()}")
print(f"Columns: {len(df.columns)}")
print(f"Date range:")
display(df.selectExpr("min(ref_date)", "max(ref_date)"))


In [0]:

# How many rows per NAICS code
print("Rows per NAICS code:")
display(df.groupBy("naics_code", "naics_description")
    .count()
    .orderBy("naics_code"))



In [0]:
# How many suppressed values (status = 'x')
print("Suppressed values:")
display(df.groupBy("status").count())

In [0]:
display(df.groupBy("geo").count().orderBy("geo"))


In [0]:
# Check suppressed values by province
# Some smaller provinces may be heavily suppressed
from pyspark.sql.functions import col

display(df.filter(col("status") == "x")
    .groupBy("geo")
    .count()
    .orderBy("count", ascending=False))

In [0]:
# notebooks/01_demand_sensing/02_bronze_to_silver.py

from pyspark.sql.functions import col, sum as spark_sum

TARGET_NAICS = ["445", "456", "457", "458", "455"]

MAJOR_PROVINCES = [
    "Ontario", "Quebec", "British Columbia",
    "Alberta", "Manitoba", "Saskatchewan"
]

CLEAN_STATUSES = ["A", "B", "C", "D", "E", None]

df = spark.table("cpg_planning.bronze.demand_statcan_retail")

silver = (df
    .filter(col("naics_code").isin(TARGET_NAICS))
    .filter(col("geo").isin(MAJOR_PROVINCES))
    .filter(col("status").isin(CLEAN_STATUSES) | col("status").isNull())
    .select("ref_date", "geo", "naics_code", "naics_description", "value", "status"))

# Validate before writing
print(f"Silver rows: {silver.count()}")
display(silver.groupBy("naics_code", "geo").count().orderBy("naics_code", "geo"))

In [0]:
   (silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cpg_planning.silver.demand_retail_monthly"))

# Validate
count = spark.table("cpg_planning.silver.demand_retail_monthly").count()
print(f"Silver table rows: {count}")